# SSE Chatbot with Reconnection with htmx v4

## Notes

1. **Auto-reconnection with `hx_sse_connect`**: Unlike `hx_get`, `hx_sse_connect` enables auto-reconnect with exponential backoff by default. No manual config needed.

2. **Resuming with `Last-Event-ID`**: On reconnect, the browser sends the last received `id:` value as a `Last-Event-ID` header. Your server can read this to resume from where the client left off.

3. **Closing the connection**: Two approaches:
   - **`hx_sse_close="event_name"`**: Server sends a named event (e.g. `event: done\ndata:\n\n`), client closes gracefully on receipt
   - **Remove/replace the element**: Removing the DOM element with `hx_sse_connect` from the page also closes the connection.

4. **Multiline SSE data**: SSE requires every line of content to have its own `data:` prefix. If your content contains newlines, split and prefix each line:
   ```python
   data_lines = '\n'.join(f'data: {line}' for line in content.splitlines())
   yield f"id: {msg_id}\n{data_lines}\n\n"
   ```
   Without this, the SSE parser silently drops everything after the first newline.

In [1]:
from fasthtml.common import *
from fasthtml.jupyter import *

In [2]:
import json
from fastcore.meta import use_kwargs, delegates
import asyncio
from claudette import Client as ClaudetteClient
from claudette import models
from starlette.responses import StreamingResponse
from asyncio import create_task

cli = ClaudetteClient(models[1])
sp = """You are a helpful and concise assistant."""
messages = []

tlink = (Script(src="https://cdn.tailwindcss.com"),)
dlink = Link(
    rel="stylesheet",
    href="https://cdn.jsdelivr.net/npm/daisyui@4.11.1/dist/full.min.css",
)

app, rt = fast_app(live=True, htmx=False, htmx4=True, hdrs=(tlink, dlink, picolink), exts='sse')

def ChatMessage(msg_idx, streaming=False, **kwargs):
    msg = messages[msg_idx]
    bubble_class = "chat-bubble-primary" if msg["role"] == "user" else "chat-bubble-secondary"
    chat_class = "chat-end" if msg["role"] == "user" else "chat-start"
    stream_attrs = dict(
        hx_sse_connect=f"/get-message?msg_idx={msg_idx}",
        hx_swap="beforeend show:bottom",
        hx_sse_close="done"
    ) if streaming else {}
    return Div(
        Div(msg["role"], cls="chat-header"),
        Div(msg["content"], id=f"chat-content-{msg_idx}", cls=f"chat-bubble {bubble_class}", **stream_attrs, **kwargs),
        id=f"chat-message-{msg_idx}",
        cls=f"chat {chat_class}")

def ChatInput():
    return Input(
        type="text", name="msg", id="msg-input",
        placeholder="Type a message",
        cls="input input-bordered w-full",
        hx_swap_oob="true",
    )

@app.route("/")
def get():
    page = Body(
        H1("Chatbot SSE (server-sent events) Demo"),
        Div(
            *[ChatMessage(i) for i in range(len(messages))],
            id="chatlist",
            cls="chat-box h-[73vh] overflow-y-auto",
        ),
        Form(
            Group(ChatInput(), Button("Send", cls="btn btn-primary")),
            hx_post="/send-message",
            hx_target="#chatlist",
            hx_swap="beforeend",
            cls="flex space-x-2 mt-2",
        ),
        cls="p-4 max-w-lg mx-auto",
    )
    return Title("Chatbot Demo"), page

active_streams = {}

async def fetch_response(msg_idx):
    r = cli(messages[:-1], sp=sp, stream=True)
    for chunk in r:
        messages[msg_idx]["content"] += chunk
        await asyncio.sleep(0.1)
    active_streams.pop(msg_idx, None)

@app.post("/send-message")
async def send_message(msg: str):
    messages.append({"role": "user", "content": msg})
    user_msg = Div(ChatMessage(len(messages) - 1))
    messages.append({"role": "assistant", "content": ""})
    msg_idx = len(messages) - 1
    active_streams[msg_idx] = True
    create_task(fetch_response(msg_idx))
    assistant_msg = Div(ChatMessage(msg_idx, streaming=True))
    return user_msg, assistant_msg, ChatInput()

async def message_generator(msg_idx, last_id=0):
    last_sent = last_id
    while msg_idx in active_streams or last_sent < len(messages[msg_idx]["content"]):
        content = messages[msg_idx]["content"]
        if len(content) > last_sent:
            new_content = content[last_sent:]
            last_sent = len(content)
            data_lines = '\n'.join(f'data: {line}' for line in new_content.splitlines())
            yield f"id: {last_sent}\n{data_lines}\n\n"
        await asyncio.sleep(0.1)
    yield "event: done\ndata:\n\n"

@app.get("/get-message")
async def get_message(request, msg_idx: int):
    last_id = int(request.headers.get("Last-Event-ID", 0))
    return StreamingResponse(message_generator(msg_idx, last_id), media_type="text/event-stream")

In [3]:
srv = JupyUvi(app)